In [1]:
#get root path and run setup file
import sys,os
root_path = os.path.abspath(os.path.join('..', 'mastodon-sim', 'src', 'sim', 'analysis_utils', 'dashboard'))
print(root_path)
if root_path not in sys.path:
    sys.path.append(root_path)

# computation
from IPython.display import Math
from IPython.display import display, HTML
from importlib import reload
import itertools
from functools import partial
from copy import deepcopy
import time
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
from itertools import product
from scipy import special
from shutil import which
import os
# plotting
import matplotlib.pyplot as pl
import seaborn as sns
import networkx as nx
from data_processing import (
    post_process_output,
    get_toot_dict,
    get_int_dict,
    get_act_dict
)
sns.set_style("ticks", {'axes.grid': True})
pl.rc("figure", facecolor="white", figsize=(8, 8))
#pl.rc("figure", facecolor="gray",figsize = (8,8))
if which('latex'):
    pl.rc('text', usetex=True)
    pl.rc('text.latex', preamble=r'\usepackage{amsmath}')
pl.rc('lines', markeredgewidth=2)
pl.rc('font', size=10)

# notebook config
display(HTML("<style>.container { width:100% !important; }</style>"))


%load_ext autoreload
%autoreload 2
%matplotlib inline

/mnt/c/Users/maxpu/Dropbox/scripts/Projects/socialsandbox/mastodon-sim/src/sim/analysis_utils/dashboard


In [4]:

# Probe configuration
PROBE_LABEL = "VotePref"

def load_p_and_t_data_from_folder(folder_path):
    # Construct full paths
    p_and_t_path = os.path.join(folder_path, "prompts_and_responses.jsonl")

    # Validation
    if not os.path.exists(p_and_t_path):
        raise ValueError(f"Missing: {p_and_t_path}")

    # Load dataframes directly from disk
    pt_df = pd.read_json(p_and_t_path, lines=True)

    return pt_df

def load_data_from_folder(folder_path):
    # Construct full paths
    action_path = os.path.join(folder_path, "mastodon_action_events.jsonl")
    probe_path = os.path.join(folder_path, "probe_events.jsonl")

    # Validation
    if not os.path.exists(action_path):
        raise ValueError(f"Missing: {action_path}")
    if not os.path.exists(probe_path):
        raise ValueError(f"Missing: {probe_path}")

    # Load dataframes directly from disk
    action_df = pd.read_json(action_path, lines=True)
    probe_df = pd.read_json(probe_path, lines=True)

    return pd.concat([action_df, probe_df], ignore_index=True)

def post_process(df):
    # Ensure all toot_ids are strings
    def get_toot_id(data):
        if "toot_id" in data:
            data["toot_id"] = str(data["toot_id"])
        return data

    df["data"] = df.data.apply(get_toot_id)

    # Process dataframes
    probe_df_processed, int_df, edge_df, act_df = post_process_output(df)

    # Extract probe data
    num_entries = len(probe_df_processed.loc[probe_df_processed.label == PROBE_LABEL])
    print(f"{num_entries} probe entries!")

    probe_data = (
        probe_df_processed.loc[
            probe_df_processed.label == PROBE_LABEL, ["source_user", "response", "episode"]
        ]
        .groupby("episode")
        .apply(lambda x: dict(zip(x.source_user, x.response, strict=False)))
        .to_dict()
    )

    # Build follow network
    follow_graph = nx.from_pandas_edgelist(
        edge_df, "source_user", "target_user", create_using=nx.DiGraph()
    )

    # Get active users by episode
    active_users_by_episode = int_df.groupby("episode")["source_user"].apply(set).to_dict()

    # Get toot and interaction data
    toot_dict, toot_owner_dict = get_toot_dict(int_df.copy())
    int_dict = get_int_dict(int_df.copy(), toot_owner_dict)

    # Get action data
    act_dict = get_act_dict(act_df.copy())

    return (
        follow_graph,
        int_dict,
        active_users_by_episode,
        toot_dict,
        probe_data,
        act_dict,
    )

In [22]:
# filename = "N20_T2_Reddit.Big5_independent_v1_news_bill_bias_None_run1_2025-12-16_19-15-59"
# filename = "N20_T2_Reddit.Big5_independent_v1_news_bill_bias_None_run1_2025-12-17_11-57-39"
filename = "N20_T5_Reddit.Big5_independent_v1_news_bill_bias_None_run1_2025-12-18_15-26-07"

folder_path = "/mnt/c/Users/maxpu/Dropbox/scripts/Projects/socialsandbox/mastodon-sim/src/scenarios/election/outputs/N20_T5_Reddit.Big5_independent_v1_news_bill_bias_None_run1/"

df = load_data_from_folder(folder_path+filename)
(
    follow_graph,
    int_dict,
    active_users_by_episode,
    toot_dict,
    probe_data,
    act_dict,
) = post_process(df)

        # except KeyError:
        #     print("KeyError:")
        #     print(row.data)
        #     exit()


84 probe entries!


In [23]:
pt_df=load_p_and_t_data_from_folder(folder_path+filename)

In [24]:
print(pt_df.prompt[30])
print(pt_df.output[30])

Role playing instructions:
The instructions for how to play the role of Lisa Martinez are as follows. This is a social science experiment studying how well you play the role of a character named Lisa Martinez. The experiment is structured as a tabletop roleplaying game (like dungeons and dragons). However, in this case it is a serious social science experiment and simulation. The goal is to be realistic. It is important to play the role of a person like Lisa Martinez as accurately as possible, i.e., by responding in ways that you think it is likely a person like Lisa Martinez would respond, and taking into account all information about Lisa Martinez that you have. Always use third-person limited perspective.

Overarching goal:
Their goal is have a good day and vote in the election.

Scenario Information:
Bill Fredrickson campaigns on providing tax breaks to local industry and creating jobs to help grow the economy.. 
Bradley Carter campaigns on increasing regulation to protect the envi

In [20]:
df.label.value_counts()

label
follow              213
Favorability        168
VotePref             84
VoteIntent           84
reply                70
get_own_timeline     21
post                 21
update_profile       21
like_toot            10
Name: count, dtype: int64

In [21]:
names_of_focalplayers = ["Bill Fredrickson", "Bradley Carter"]

pd.set_option("display.width", 1000)
print(df.head())
print()
print("Probes:")
print(df.loc[df.event_type == "probe", "label"].value_counts())
print()
print("Actions:")
print(df.loc[(df.event_type == "action") & (df.episode > -1), "label"].value_counts())
print()
print(
    df.loc[(df.event_type == "action") & (df.episode > -1) & df.data.apply(lambda x: type(x)==dict), "data"]
    .apply(lambda x: x.keys())
    .value_counts()
)
print()
print(
    df.loc[
        (df.source_user.isin(names_of_focalplayers))
        & (df.event_type == "action")
        & (df.episode > -1)
    ]
    .groupby("source_user")["label"]
    .value_counts()
)
print()
print("post")
posts = df.loc[
    (df.label == "post") & (df.episode > -1) & (df.source_user != "storhampton_gazette"),
    ["source_user", "data"],
]
users = posts["source_user"].values
posts = posts["data"].apply(lambda x: x["post_text"]).values
for user, post in zip(users, posts):
    print(f"{user}:\t\t{post}")
print()
print("reply")
replies = df.loc[
    (df.label == "reply") & (df.episode > -1) & (df.source_user != "storhampton_gazette"),
    ["source_user", "episode", "data"],
]
users = replies["source_user"].values
replies = replies["data"].apply(lambda x: x["post_text"]).values
for user, reply in zip(users, replies):
    print(f"source_user:{user}:{reply}")
print()
dftmp = df.copy()
dftmp = dftmp.loc[(dftmp.event_type == "action") & (dftmp.episode > -1), :]
dftmp["data"] = dftmp["data"].apply(str)
print(f"{(len(dftmp) - len(dftmp.drop_duplicates())) / len(dftmp):.3f} duplicate fraction")


        source_user   label                                               data  episode event_type
0  Michael Thompson  follow  {'target_user': 'Michael Donovan', 'suggested_...        0     action
1   Michael Donovan  follow  {'target_user': 'Jessica Lopez', 'suggested_ac...        0     action
2   Rachel Thompson  follow  {'target_user': 'Michael Donovan', 'suggested_...        0     action
3        Emily Chen  follow  {'target_user': 'Michael Donovan', 'suggested_...        0     action
4       Arjun Patel  follow  {'target_user': 'Rachel Thompson', 'suggested_...        0     action

Probes:
label
Favorability    168
VotePref         84
VoteIntent       84
Name: count, dtype: int64

Actions:
label
follow              213
reply                70
update_profile       21
post                 21
get_own_timeline     21
like_toot            10
Name: count, dtype: int64

data
(target_user, suggested_action)                     213
(reply_to, toot_id, post_text, suggested_action)     70
(